# Data Access

> Functionality to access Euclid calibrated data frames for specific targets and observation IDs.

In [ ]:
# | default_exp euclid.data_access

In [ ]:
# | export

from astropy import table
from astropy.table import Table
from astroquery.utils.tap.core import TapPlus
from getpass import getpass

import os

import numpy as np

In [ ]:
# | hide
from nbdev import show_doc

In [ ]:
# | export


class DataAccess:
    """Provides access to Euclid data."""

    def __init__(
        self,
        esa_username=None,  # ESA account username (prompts if not supplied)
        esa_password=None,  # ESA account password (prompts if not supplied)
        esac_server_url="https://easidr.esac.esa.int",  # ESA server (default is Q1)
        release_name="Q1_R1",  # the Euclid release name
        dry_run=False,  # if True, do not actually download files
    ):
        """Create an object for accessing data and log in to the ESA server."""
        if esa_username is None:
            self.esa_username = getpass(prompt="ESA User:")
        else:
            self.esa_username = esa_username
        if esa_password is None:
            self.esa_password = getpass(prompt="ESA Password:")
        else:
            self.esa_password = esa_password
        self.release_name = release_name
        self.dry_run = dry_run
        self.tap = TapPlus(url=f"{esac_server_url}/tap-server", tap_context="tap")
        self.data_tap = TapPlus(url=f"{esac_server_url}/sas-dd", data_context="data")

    def tap_login(self):
        self.tap.login(user=self.esa_username, password=self.esa_password)

    def data_login(self):
        self.data_tap.login(user=self.esa_username, password=self.esa_password)

    def tap_query(self, query):
        self.tap_login()
        job = self.tap.launch_job(query)
        return job.get_results()

    def build_instrument_condition(self, instrument, filter, raw=False):
        release_condition = "1=1" if raw else f"(release_name='{self.release_name}')" 
        instrument_condition = (
            f"AND instrument_name = '{instrument}'" if instrument is not None else ""
        )
        filter_condition = f"AND filter_name = '{filter}'" if filter is not None else ""
        return " ".join((release_condition, instrument_condition, filter_condition))

    def build_fov_condition(self, ra, dec, radius, fully_contained):
        release_condition = f"(release_name='{self.release_name}')"
        criterion = "CONTAINS" if fully_contained else "INTERSECTS"
        fov_condition = f"AND (fov IS NOT NULL AND {criterion}(CIRCLE('ICRS',{ra},{dec},{radius}),fov)=1)"
        return " ".join((release_condition, fov_condition))

    def find_observations_for_target(
        self,
        ra,  # RA of the target, in decimal degrees
        dec,  # Dec of the target, in decimal degrees
        radius=1 / 60,  # radius of the target, in decimal degrees
        fully_contained=True,  # if False, the target region only needs to intersect with the observation footprint
    ):  # returns a list of observation_ids
        """Obtain a list of survey obs_ids for observations that entirely contain or intersect the specified target region."""
        condition = self.build_fov_condition(ra, dec, radius, fully_contained)
        query = f"""SELECT observation_id
                    FROM sedm.calibrated_frame
                    WHERE (product_type like '%Calibrated%')
                    AND {condition}
                    ORDER BY observation_id ASC"""
        results = self.tap_query(query)
        obs_ids = np.unique(list(results["observation_id"])).astype(int)
        return obs_ids

    def find_tiles_for_target(
        self,
        ra,  # RA of the target, in decimal degrees
        dec,  # Dec of the target, in decimal degrees
        radius=1 / 60,  # radius of the target, in decimal degrees
        fully_contained=True,  # if False, the target region only needs to intersect with the observation footprint
    ):  # returns a list of tile_indexes
        """Obtain a list of survey MER tile_indexes for tiles that entirely contain or intersect the specified target region."""
        condition = self.build_fov_condition(ra, dec, radius, fully_contained)
        query = f"""SELECT tile_index
                    FROM sedm.mosaic_product
                    WHERE {condition}
                    ORDER BY tile_index ASC"""
        results = self.tap_query(query)
        tile_indexes = np.unique(list(results["tile_index"])).astype(int)
        return tile_indexes
    
    def get_calibrated_files_for_observation(
        self,
        obs_id,  # observation_id for which to find files
        instrument=None,  # None, NISP or VIS
        filter=None,  # None, VIS, NIR_Y, NIR_J or NIR_H
    ):  # returns a table of file information
        """Obtain calibrated file information for obs_id, optionally restricted by instrument or filter."""
        condition = self.build_instrument_condition(instrument, filter)
        query = f"""SELECT observation_stack.observation_id, observation_stack.file_name,
                    observation_stack.instrument_name, observation_stack.filter_name, observation_stack.duration
                    FROM sedm.calibrated_frame
                    AS observation_stack
                    WHERE (product_type like '%Calibrated%')
                    AND {condition}
                    AND (observation_id = '{obs_id}')"""
        file_info = self.tap_query(query)
        return file_info

    def get_raw_files_for_observation(
        self,
        obs_id,  # observation_id for which to find files
        instrument=None,  # None, NISP or VIS
        filter=None,  # None, "" for VIS, OPEN for grism, CLOSED for slew dark, Y, J or H
    ):  # returns a table of file information
        """Obtain raw file information for obs_id, optionally restricted by instrument or filter."""
        condition = self.build_instrument_condition(instrument, filter, raw=True)
        query = f"""SELECT raw_frame.file_name, raw_frame.rawframe_oid, raw_frame.observation_id,
                    raw_frame.instrument_name, raw_frame.data_set_release, raw_frame.filter_name,
                    raw_frame.observation_mode, raw_frame.grism_wheel_pos, raw_frame.cal_block_id,
                    raw_frame.cal_block_variant, raw_frame.ra, raw_frame.dec, raw_frame.obs_time_utc,
                    raw_frame.exposure_time
                    FROM sedm.raw_frame
                    WHERE {condition}
                    AND (observation_id = '{obs_id}')"""
        # (observation_mode='{LE1type}')
        file_info = self.tap_query(query)
        return file_info

    def get_files_for_tile(
        self,
        tile_index,  # tile_index for which to find files
        instrument=None,  # None, NISP or VIS
        filter=None,  # None, VIS, NIR_Y, NIR_J or NIR_H
    ):  # returns a table of file information
        """Obtain mosaic file information for tile_index, optionally restricted by instrument or filter."""
        condition = self.build_instrument_condition(instrument, filter)
        query = f"""SELECT mosaic_product.tile_index, mosaic_product.file_name, mosaic_product.mosaic_product_oid,
                    mosaic_product.instrument_name, mosaic_product.filter_name, mosaic_product.category,
                    mosaic_product.second_type, mosaic_product.ra, mosaic_product.dec, mosaic_product.technique 
                    FROM sedm.mosaic_product
                    WHERE {condition}
                    AND (tile_index = '{tile_index}')"""
        file_info = self.tap_query(query)
        return file_info
    
    def download_files(
        self,
        filenames,  #  list of filenames or Table containing "file_name" column
        outpath="./",  # the folder in which to save the downloaded files
        verbose=True,  # print information to the screen
    ):
        """Download multiple Euclid filenames to outpath."""
        if isinstance(filenames, Table):
            filenames = filenames["file_name"]
        if verbose:
            print(f"Downloading {len(filenames)} files to {outpath}")
        for fn in filenames:
            if verbose:
                print(f"Downloading {fn}")
            self.download_file(fn, outpath)

    def download_file(
        self,
        filename,  # the filename to download
        outpath="./",  # the folder in which to save the downloaded files
    ):
        """Download Euclid filename to outpath."""
        params_dict = dict(
            RETRIEVAL_TYPE="FILE", RELEASE="sedm", RETRIEVAL_ACCESS="DIRECT"
        )
        params_dict.update(FILE_NAME=filename)
        outpath = os.path.expanduser(outpath)
        outfn = os.path.join(outpath, filename)
        self.data_login()
        if not self.dry_run:
            self.data_tap.load_data(params_dict=params_dict, output_file=outfn)

    def download_mosaic_files(
        self,
        file_info,  #  Table containing file information
        mer_file_type="STK",  # STK, BKG, RMS or FLAG
        outpath="./",  # the folder in which to save the downloaded files
        verbose=True,  # print information to the screen
    ):
        """Download multiple Euclid mosaics to outpath, using an alternative method."""
        print(f"Downloading {len(file_info)} files to {outpath}")
        for info in file_info:
            t = info["tile_index"]
            i = info["instrument_name"]
            f = info["filter_name"]
            self.download_mosaic_file(t, i, f, mer_file_type, outpath=outpath, verbose=verbose)

    def download_mosaic_file(
        self,
        tile_index,
        instrument,  # NISP or VIS
        filter,  # VIS, NIR_Y, NIR_J or NIR_H
        mer_file_type="STK",  # STK, BKG, RMS or FLAG
        outpath="./",  # the folder in which to save the downloaded files
        verbose=True,  # print information to the screen
    ):
        """Download Euclid mosaic to outpath, using an alternative method."""
        params_dict = dict(
            RETRIEVAL_TYPE="MOSAIC", RELEASE="sedm"
        )
        params_dict.update(TILE_INDEX=tile_index, INSTRUMENT=instrument, FILTER=filter, TYPE=mer_file_type)
        outpath = os.path.expanduser(outpath)
        if mer_file_type == "STK":
            filename = f"BGSUB-MOSAIC-{filter}"
        elif mer_file_type == "BKG":
            filename = f"BGMOD-{filter}"
        elif mer_file_type == "RMS":
            filename = f"MOSAIC-{filter}-RMS"
        elif mer_file_type == "FLAG":
            filename = f"MOSAIC-{filter}-FLAG"
        else:
            raise ValueError(f"Invalid mer_file_type provided: {mer_file_type}.")
        filename = f"EUC_MER_{filename}_TILE{tile_index}.fits"
        outfn = os.path.join(outpath, filename)
        if verbose:
            print(f"Downloading {mer_file_type} file for tile {tile_index}, instrument {instrument} and filter {filter}")
            print(f"Saving as {outfn}")
            print(params_dict)
        self.data_login()
        if not self.dry_run:
            self.data_tap.load_data(params_dict=params_dict, output_file=outfn)

    def download_calibrated_files_for_observation(
        self,
        obs_id,
        outpath="./",  # the folder in which to save the downloaded files
        instrument=None,  # None, NISP or VIS
        filter=None,  # None, VIS, NIR_Y, NIR_J or NIR_H
        verbose=True,  # print information to the screen
    ):  #  returns a table of file information
        """Download all calibrated files for a Euclid observation, optionally restricted by instrument or filter."""
        file_info = self.get_calibrated_files_for_observation(
            obs_id, instrument=instrument, filter=filter
        )
        self.download_files(file_info, outpath=outpath, verbose=verbose)
        return file_info

    def download_raw_files_for_observation(
        self,
        obs_id,
        outpath="./",  # the folder in which to save the downloaded files
        instrument=None,  # None, NISP or VIS
        filter=None,  # None, VIS, NIR_Y, NIR_J or NIR_H
        verbose=True,  # print information to the screen
    ):  #  returns a table of file information
        """Download all calibrated files for a Euclid observation, optionally restricted by instrument or filter."""
        file_info = self.get_raw_files_for_observation(
            obs_id, instrument=instrument, filter=filter
        )
        self.download_files(file_info, outpath=outpath, verbose=verbose)
        return file_info
    
    def download_files_for_tile(
        self,
        tile_index,
        outpath="./",  # the folder in which to save the downloaded files
        instrument=None,  # None, NISP or VIS
        filter=None,  # None, VIS, NIR_Y, NIR_J or NIR_H
        mer_file_type="STK",  # "STK", BKG, RMS or FLAG
        verbose=True,  # print information to the screen
    ):  #  returns a table of file information
        """Download all files for a Euclid MER tile, optionally restricted by instrument or filter.
        
        By default this gets the background subtracted image, but specifying `mer_file_type` can download the
        background model (BKG), RMS or FLAG images.
        """
        file_info = self.get_files_for_tile(
            tile_index, instrument=instrument, filter=filter
        )
        #self.download_mosaic_files(file_info, mer_file_type, outpath=outpath, verbose=verbose)
        self.download_files(file_info, outpath=outpath, verbose=verbose)
        return file_info
    
    def download_files_for_target(
        self,
        ra,  # RA of the target, in decimal degrees
        dec,  # Dec of the target, in decimal degrees
        radius=1 / 60,  # radius of the target, in decimal degrees
        fully_contained=True,  # if False, the target region only needs to intersect with the observation footprint
        outpath="./",  # the folder in which to save the downloaded files
        instrument=None,  # None, NISP or VIS
        filter=None,  # None, VIS, NIR_Y, NIR_J or NIR_H
        file_type="CAL",  # CAL, MER or LE1
        mer_file_type="STK",  # STK, BKG, RMS or FLAG, only for file_type="MER"
        verbose=True,  # print information to the screen
    ):  #  returns a table of file information
        """Download all calibrated files for Euclid observations covering a target, optionally restricted by instrument or filter."""
        file_info = []
        if file_type == "CAL":
            obs_ids = self.find_observations_for_target(
                ra, dec, radius, fully_contained=fully_contained
            )
            for obs_id in obs_ids:
                if verbose:
                    print(f"Downloading files for observation id {obs_id}")
                obs_file_info = self.download_calibrated_files_for_observation(
                    obs_id,
                    outpath=outpath,
                    instrument=instrument,
                    filter=filter,
                    verbose=verbose,
                )
                file_info.append(obs_file_info)
        elif file_type == "MER":
            tile_ids = self.find_tiles_for_target(
                ra, dec, radius, fully_contained=fully_contained
            )
            for tile_id in tile_ids:
                if verbose:
                    print(f"Downloading files for observation id {tile_id}")
                tile_file_info = self.download_files_for_tile(
                    tile_id,
                    outpath=outpath,
                    instrument=instrument,
                    filter=filter,
                    mer_file_type=mer_file_type,
                    verbose=verbose,
                )
                file_info.append(tile_file_info)
        if len(file_info) > 0:
            return table.vstack(file_info)

In [ ]:
show_doc(DataAccess.find_observations_for_target)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L69){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.find_observations_for_target

>      DataAccess.find_observations_for_target (ra, dec,
>                                               radius=0.016666666666666666,
>                                               fully_contained=True)

*Obtain a list of survey obs_ids for observations that entirely contain or intersect the specified target region.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| ra |  |  | RA of the target, in decimal degrees |
| dec |  |  | Dec of the target, in decimal degrees |
| radius | float | 0.016666666666666666 | radius of the target, in decimal degrees |
| fully_contained | bool | True | if False, the target region only needs to intersect with the observation footprint |

In [ ]:
show_doc(DataAccess.get_calibrated_files_for_observation)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L104){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.get_calibrated_files_for_observation

>      DataAccess.get_calibrated_files_for_observation (obs_id, instrument=None,
>                                                       filter=None)

*Obtain calibrated file information for obs_id, optionally restricted by instrument or filter.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| obs_id |  |  | observation_id for which to find files |
| instrument | NoneType | None | None, NISP or VIS |
| filter | NoneType | None | None, VIS, NIR_Y, NIR_J or NIR_H |

In [ ]:
show_doc(DataAccess.download_calibrated_files_for_observation)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L221){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.download_calibrated_files_for_observation

>      DataAccess.download_calibrated_files_for_observation (obs_id,
>                                                            outpath='./',
>                                                            instrument=None,
>                                                            filter=None,
>                                                            verbose=True)

*Download all calibrated files for a Euclid observation, optionally restricted by instrument or filter.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| obs_id |  |  |  |
| outpath | str | ./ | the folder in which to save the downloaded files |
| instrument | NoneType | None | None, NISP or VIS |
| filter | NoneType | None | None, VIS, NIR_Y, NIR_J or NIR_H |
| verbose | bool | True | print information to the screen |

In [ ]:
show_doc(DataAccess.download_files_for_target)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L257){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.download_files_for_target

>      DataAccess.download_files_for_target (ra, dec,
>                                            radius=0.016666666666666666,
>                                            fully_contained=True, outpath='./',
>                                            instrument=None, filter=None,
>                                            file_type='CAL',
>                                            mer_file_type='STK', verbose=True)

*Download all calibrated files for Euclid observations covering a target, optionally restricted by instrument or filter.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| ra |  |  | RA of the target, in decimal degrees |
| dec |  |  | Dec of the target, in decimal degrees |
| radius | float | 0.016666666666666666 | radius of the target, in decimal degrees |
| fully_contained | bool | True | if False, the target region only needs to intersect with the observation footprint |
| outpath | str | ./ | the folder in which to save the downloaded files |
| instrument | NoneType | None | None, NISP or VIS |
| filter | NoneType | None | None, VIS, NIR_Y, NIR_J or NIR_H |
| file_type | str | CAL | CAL, MER or LE1 |
| mer_file_type | str | STK | STK, BKG, RMS or FLAG, only for file_type="MER" |
| verbose | bool | True | print information to the screen |

### Example

The following demonstrates use of the DataAccess class. This would normally first be imported using `from nicl.euclid.data_access import DataAccess`.

To actually download files, remove `dry_run=True`.

In [ ]:
#da = DataAccess(dry_run=True)
da = DataAccess(dry_run=True, esa_username="sbamford", esa_password="4WvAZpCYv@ktdzrW")

#### Calibrated frames

In [ ]:
da.find_observations_for_target(ra=265.2125, dec=67.3972)

INFO: OK [astroquery.utils.tap.core]


array([2681])

In [ ]:
da.find_observations_for_target(
    ra=265.2125, dec=67.3972, radius=1.0, fully_contained=False
)

INFO: OK [astroquery.utils.tap.core]


array([2681, 2682, 2686, 2687, 2688, 2691])

In [ ]:
da.get_calibrated_files_for_observation(2681, instrument="NISP", filter="NIR_H")

INFO: OK [astroquery.utils.tap.core]


observation_id,file_name,instrument_name,filter_name,duration
str255,str255,str255,str255,float64
2681,EUC_NIR_W-CAL-IMAGE_H-2681-3_20240930T175003.801620Z.fits,NISP,NIR_H,87.2448
2681,EUC_NIR_W-CAL-IMAGE_H-2681-1_20240930T175003.573405Z.fits,NISP,NIR_H,87.2448
2681,EUC_NIR_W-CAL-IMAGE_H-2681-0_20240930T175003.578743Z.fits,NISP,NIR_H,87.2448
2681,EUC_NIR_W-CAL-IMAGE_H-2681-2_20240930T175003.453972Z.fits,NISP,NIR_H,87.2448


In [ ]:
da.download_calibrated_files_for_observation(
    2681, instrument="NISP", filter="NIR_H", outpath="~/data/euclid/test/"
)

INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]


observation_id,file_name,instrument_name,filter_name,duration
str255,str255,str255,str255,float64
2681,EUC_NIR_W-CAL-IMAGE_H-2681-1_20240930T175003.573405Z.fits,NISP,NIR_H,87.2448
2681,EUC_NIR_W-CAL-IMAGE_H-2681-3_20240930T175003.801620Z.fits,NISP,NIR_H,87.2448
2681,EUC_NIR_W-CAL-IMAGE_H-2681-2_20240930T175003.453972Z.fits,NISP,NIR_H,87.2448
2681,EUC_NIR_W-CAL-IMAGE_H-2681-0_20240930T175003.578743Z.fits,NISP,NIR_H,87.2448


In [ ]:
da.download_files_for_target(
    ra=265.2125, dec=67.3972, instrument="NISP", filter="NIR_H", file_type="CAL", outpath="~/data/euclid/test/"
)

INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]


observation_id,file_name,instrument_name,filter_name,duration
str255,str255,str255,str255,float64
2681,EUC_NIR_W-CAL-IMAGE_H-2681-3_20240930T175003.801620Z.fits,NISP,NIR_H,87.2448
2681,EUC_NIR_W-CAL-IMAGE_H-2681-1_20240930T175003.573405Z.fits,NISP,NIR_H,87.2448
2681,EUC_NIR_W-CAL-IMAGE_H-2681-2_20240930T175003.453972Z.fits,NISP,NIR_H,87.2448
2681,EUC_NIR_W-CAL-IMAGE_H-2681-0_20240930T175003.578743Z.fits,NISP,NIR_H,87.2448


#### MER tiles

::: {.callout-warning}
Note that currently (at 2024-11-20) there does not appear to be any way to programatically download the MER auxilliary files, e.g., BKG, RMS, FLAG. The documented methods are broken. Instead, to download these files, use a browser to search the [SAS](https://easidr.esac.esa.int/sas/) and use the bulk download link on the results page.
:::

In [ ]:
da.find_tiles_for_target(ra=265.2125, dec=67.3972)

INFO: OK [astroquery.utils.tap.core]


array([102160055])

In [ ]:
da.find_tiles_for_target(
    ra=265.2125, dec=67.3972, radius=1.0, fully_contained=False
)

INFO: OK [astroquery.utils.tap.core]


array([102159481, 102159482, 102159483, 102159484, 102159770, 102159771,
       102159772, 102159773, 102159774, 102160054, 102160055, 102160056,
       102160057, 102160333, 102160334, 102160335, 102160606])

In [ ]:
da.get_files_for_tile(102160055, instrument="NISP", filter="NIR_H")

INFO: OK [astroquery.utils.tap.core]


tile_index,file_name,mosaic_product_oid,instrument_name,filter_name,category,second_type,ra,dec,technique
int64,str255,int64,str255,str255,str255,str255,float64,float64,str255
102160055,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160055-19F743_20241024T210738.211057Z_00.00.fits,2111,NISP,NIR_H,SCIENCE,SKY,264.8880329,67.5,IMAGE


In [ ]:
da.download_files_for_tile(
    102160055, instrument="NISP", filter="NIR_H", outpath="~/data/euclid/test/"
)

INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]


tile_index,file_name,mosaic_product_oid,instrument_name,filter_name,category,second_type,ra,dec,technique
int64,str255,int64,str255,str255,str255,str255,float64,float64,str255
102160055,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160055-19F743_20241024T210738.211057Z_00.00.fits,2111,NISP,NIR_H,SCIENCE,SKY,264.8880329,67.5,IMAGE


In [ ]:
da.download_files_for_target(
    ra=265.2125, dec=67.3972, instrument="NISP", filter="NIR_H", file_type="MER", outpath="~/data/euclid/test/"
)

INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]


tile_index,file_name,mosaic_product_oid,instrument_name,filter_name,category,second_type,ra,dec,technique
int64,str255,int64,str255,str255,str255,str255,float64,float64,str255
102160055,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160055-19F743_20241024T210738.211057Z_00.00.fits,2111,NISP,NIR_H,SCIENCE,SKY,264.8880329,67.5,IMAGE


#### Raw frames

For example, we can download the raw slew darks.

In [ ]:
da.get_raw_files_for_observation(2681, instrument="NISP", filter="CLOSED")

INFO: OK [astroquery.utils.tap.core]


file_name,rawframe_oid,observation_id,instrument_name,data_set_release,filter_name,observation_mode,grism_wheel_pos,cal_block_id,cal_block_variant,ra,dec,obs_time_utc,exposure_time
str255,int64,int64,str255,str255,str255,str255,str255,str255,str255,float64,float64,object,float64
EUC_LE1_NISP-02681-1-D_20240717T165947.000000Z_01_05_01.00.fits,2300,2681,NISP,SURVEY,CLOSED,CALIBRATION,OPEN,CALBLOCK-F-003,0,265.30365681,67.4007584,2024-07-17T17:17:04.154,87.2448


In [ ]:
da.download_raw_files_for_observation(
    2681, instrument="NISP", filter="CLOSED", outpath="~/data/euclid/test/"
)

INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]


file_name,rawframe_oid,observation_id,instrument_name,data_set_release,filter_name,observation_mode,grism_wheel_pos,cal_block_id,cal_block_variant,ra,dec,obs_time_utc,exposure_time
str255,int64,int64,str255,str255,str255,str255,str255,str255,str255,float64,float64,object,float64
EUC_LE1_NISP-02681-1-D_20240717T165947.000000Z_01_05_01.00.fits,2300,2681,NISP,SURVEY,CLOSED,CALIBRATION,OPEN,CALBLOCK-F-003,0,265.30365681,67.4007584,2024-07-17T17:17:04.154,87.2448


In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()